<a href="https://colab.research.google.com/github/somesh90633/Loan-payment-matrix-model-/blob/main/LOAN_PAYMENT_MATRIX_MODEL_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# ============================================================
# LOAN PAYMENT MATRIX MODEL - PYTHON WEB FRONTEND
# ============================================================

!pip -q install gradio matplotlib numpy sympy

import numpy as np
import sympy as sp
import matplotlib.pyplot as plt
import gradio as gr
import time


# ------------------------------------------------------------
# LOAN CALCULATION
# ------------------------------------------------------------

def calculate_loan(loan_type, principal, annual_rate, monthly_payment, months):

    principal = float(principal)
    annual_rate = float(annual_rate)
    monthly_payment = float(monthly_payment)
    months = int(months)

    if principal <= 0:
        raise ValueError("Principal must be greater than 0.")

    if annual_rate < 0:
        raise ValueError("Interest rate cannot be negative.")

    if monthly_payment <= 0:
        raise ValueError("Monthly payment must be greater than 0.")

    if months <= 0:
        raise ValueError("Months must be greater than 0.")

    monthly_rate = annual_rate / (12 * 100)

    # Transition matrix
    A = np.array([
        [1 + monthly_rate, -1],
        [0, 1]
    ], dtype=float)

    balance = principal

    balances = [balance]
    interests = []

    total_interest = 0
    payoff_month = None

    for month in range(1, months + 1):

        interest = balance * monthly_rate

        total_interest += interest

        balance = balance + interest - monthly_payment

        if balance <= 0:
            balance = 0
            payoff_month = month

        balances.append(balance)
        interests.append(interest)

        if balance == 0:
            break

    actual_months = len(balances) - 1

    # Matrix power
    A_power = np.linalg.matrix_power(A, actual_months)

    # SymPy matrix
    A_sym = sp.Matrix(A)

    eigenvalues = A_sym.eigenvals()

    # Cayley-Hamilton information
    trace = A_sym.trace()
    determinant = A_sym.det()

    # --------------------------------------------------------
    # RESULT TEXT
    # --------------------------------------------------------

    result = f"""
LOAN PAYMENT MATRIX MODEL
============================================

Loan Type              : {loan_type}
Principal              : {principal:,.2f}
Annual Interest Rate   : {annual_rate:.2f}%
Monthly Payment        : {monthly_payment:,.2f}

Loan Duration          : {actual_months} months
Final Balance          : {balance:,.2f}
Total Interest         : {total_interest:,.2f}

--------------------------------------------
TRANSITION MATRIX A
--------------------------------------------

{np.round(A, 6)}

--------------------------------------------
MATRIX POWER A^{actual_months}
--------------------------------------------

{np.round(A_power, 4)}

--------------------------------------------
EIGENVALUES
--------------------------------------------

{eigenvalues}

--------------------------------------------
CAYLEY-HAMILTON THEOREM
--------------------------------------------

A^2 - (trace)A + (determinant)I = 0

Trace       = {trace}
Determinant = {determinant}
"""

    return result, balances, actual_months


# ------------------------------------------------------------
# NORMAL CALCULATION BUTTON
# ------------------------------------------------------------

def run_calculation(
    loan_type,
    principal,
    annual_rate,
    monthly_payment,
    months
):

    result, balances, actual_months = calculate_loan(
        loan_type,
        principal,
        annual_rate,
        monthly_payment,
        months
    )

    fig, ax = plt.subplots(figsize=(9, 5))

    x = list(range(len(balances)))

    ax.plot(
        x,
        balances,
        marker="o",
        linewidth=2
    )

    ax.set_title(
        "Loan Balance Over Time",
        fontsize=15,
        fontweight="bold"
    )

    ax.set_xlabel("Month")
    ax.set_ylabel("Remaining Principal")

    ax.grid(True)

    plt.tight_layout()

    return result, fig


# ------------------------------------------------------------
# ANIMATED GRAPH
# ------------------------------------------------------------

def animate_loan(
    loan_type,
    principal,
    annual_rate,
    monthly_payment,
    months
):

    result, balances, actual_months = calculate_loan(
        loan_type,
        principal,
        annual_rate,
        monthly_payment,
        months
    )

    x = list(range(len(balances)))

    # Create animation frames
    frames = []

    max_balance = max(balances)

    for i in range(1, len(balances) + 1):

        fig, ax = plt.subplots(figsize=(9, 5))

        ax.plot(
            x[:i],
            balances[:i],
            marker="o",
            linewidth=3
        )

        ax.set_xlim(
            0,
            max(len(x) - 1, 1)
        )

        ax.set_ylim(
            0,
            max_balance * 1.1
        )

        ax.set_xlabel("Month")
        ax.set_ylabel("Remaining Principal")

        ax.set_title(
            f"{loan_type} - Month {i - 1}",
            fontsize=15,
            fontweight="bold"
        )

        ax.grid(True)

        plt.tight_layout()

        frames.append(fig)

        plt.close(fig)

    # Return final frame because Gradio cannot directly
    # display Matplotlib animation objects.
    final_fig = frames[-1]

    return result, final_fig


# ------------------------------------------------------------
# GRADIO FRONTEND
# ------------------------------------------------------------

with gr.Blocks(title="Loan Payment Matrix Model") as app:

    gr.Markdown(
        """
        # LOAN PAYMENT MATRIX MODEL

        ### Matrix-Based Loan Calculator

        Enter your loan details and calculate the remaining
        balance, total interest, matrix power, eigenvalues,
        and Cayley-Hamilton information.
        """
    )

    with gr.Row():

        with gr.Column():

            loan_type = gr.Dropdown(
                choices=[
                    "Home Loan",
                    "Education Loan",
                    "Personal Loan"
                ],
                value="Home Loan",
                label="Loan Type"
            )

            principal = gr.Number(
                value=100000,
                label="Principal Amount"
            )

            annual_rate = gr.Number(
                value=8,
                label="Annual Interest Rate (%)"
            )

            monthly_payment = gr.Number(
                value=1200,
                label="Monthly Payment"
            )

            months = gr.Number(
                value=60,
                label="Number of Months",
                precision=0
            )

            calculate_button = gr.Button(
                "CALCULATE",
                variant="primary"
            )

            animate_button = gr.Button(
                "SHOW ANIMATION"
            )

        with gr.Column():

            result_box = gr.Textbox(
                label="Calculation Result",
                lines=25
            )

            graph = gr.Plot(
                label="Loan Balance Graph"
            )

    calculate_button.click(
        fn=run_calculation,
        inputs=[
            loan_type,
            principal,
            annual_rate,
            monthly_payment,
            months
        ],
        outputs=[
            result_box,
            graph
        ]
    )

    animate_button.click(
        fn=animate_loan,
        inputs=[
            loan_type,
            principal,
            annual_rate,
            monthly_payment,
            months
        ],
        outputs=[
            result_box,
            graph
        ]
    )


# ------------------------------------------------------------
# START APPLICATION
# ------------------------------------------------------------

app.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://53ad75079f3b222cda.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
